# Data Engineering and Management Workshop

This workbook contains the worked solution for the workshop. Use it to check the method, the code, and the reasoning behind each step.

The case study follows a fictional prosthetics charity and shows the core parts of a data engineering workflow: ingestion, cleaning, joins, and privacy masking.

You will work through four themes:

- Ingestion and pipelines
- Data cleaning and logic checks
- Relational storage and joins
- Cybersecurity and privacy masking

Before you start, run the mock data generation script so that `messy_prosthetics_data.csv` is in the same folder as this notebook.

## Step 0: Mock Data Setup

Run the generator script first. It creates `messy_prosthetics_data.csv` and the small reference table used later in the workbook.

## Section 1: Ingestion & Pipelines (ETL vs ELT)

ETL means **Extract, Transform, Load**. Data is validated before it reaches the destination system. In a healthcare setting, that usually reduces risk because privacy checks and validation happen before sensitive data moves on.

ELT means **Extract, Load, Transform**. Raw data lands first and is transformed later in a warehouse or lakehouse. That works well when you need scale and can separate sensitive records from direct identifiers.

In practice, a charity may use ETL for privacy-critical workflows and ELT for reporting once the data has been safely separated from direct identifiers.

In [ ]:
from pathlib import Path

import pandas as pd

csv_candidates = [
    Path('prosthetics_data.csv'),
    Path('..') / 'prosthetics_data.csv',
    Path.cwd() / 'prosthetics_data.csv',
]

for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find prosthetics_data.csv.')

df = pd.read_csv(csv_path)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Patient_ID        15 non-null     str  
 1   Full_Name         15 non-null     str  
 2   Age_Input         15 non-null     str  
 3   Implant_Date      15 non-null     str  
 4   Amputation_Cause  15 non-null     str  
 5   Device_Model_ID   15 non-null     str  
 6   Clinical_Notes    15 non-null     str  
dtypes: str(7)
memory usage: 972.0 bytes


## Section 2: Data Cleaning & Logic Challenges

Data validation checks whether values make sense before they enter downstream systems. Type cleaning and standardisation keep messy inputs from breaking joins, reports, or database loads.

In [11]:
import re

CURRENT_YEAR = 2026

def parse_age_input(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, float)) and not pd.isna(value):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())

    if numeric_value < 100:
        return numeric_value
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value
    return numeric_value

df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

,Patient_ID,Age_Input,Age_Years
0,PT-001,28,28
1,PT-002,34,34
2,PT-003,1995,31
3,PT-004,"Don't know, maybe 40",40
4,PT-005,1988,38
5,PT-006,around 50,50
6,PT-007,19,19
7,PT-008,41,41
8,PT-009,52,52
9,PT-010,1990,36


In [13]:
# 1. Strip the ordinal suffixes from the 'Implant_Date' column in the loaded df
df['Implant_Date_Clean'] = df['Implant_Date'].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)

# 2. Parse the mixed formats safely
df['Implant_Date_Clean'] = pd.to_datetime(
    df['Implant_Date_Clean'], 
    errors='coerce', 
    format='mixed', 
    dayfirst=True
).dt.strftime('%Y-%m-%d')

# 3. Clean strings
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()

df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean', 'Amputation_Cause', 'Amputation_Cause_Clean']]

,Patient_ID,Implant_Date,Implant_Date_Clean,Amputation_Cause,Amputation_Cause_Clean
0,PT-001,12/05/2024,2024-05-12,Trauma,trauma
1,PT-002,2024-06-15,2024-06-15,diabetes,diabetes
2,PT-003,May 12th 2024,2024-05-12,TRAUMA,trauma
3,PT-004,07/11/2023,2023-11-07,blast injury,blast injury
4,PT-005,2023/09/01,2023-01-09,Trauma,trauma
5,PT-006,March 3 2024,2024-03-03,diabetes,diabetes
6,PT-007,2024-01-20,2024-01-20,TRAUMA,trauma
7,PT-008,20/02/2024,2024-02-20,blast Injury,blast injury
8,PT-009,April 5th 2024,2024-04-05,trauma,trauma
9,PT-010,2024-08-30,2024-08-30,DIABETES,diabetes


## Section 3: Relational Storage & Big Data

Structured data fits into tables, rows, and columns, so SQL databases are a good fit when you need rules and relationships. Unstructured data includes notes, messages, and scanned documents.

Primary keys identify rows. Foreign keys point to related rows in other tables. Joins use those keys to connect data without duplicating it.

For this charity, patient records can be joined to device inventory so analysts can see which model was fitted, what it cost, and who made it.

In [ ]:
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite', 'Cost_USD': 950, 'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha', 'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro', 'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field', 'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}

device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
device_inventory_df = device_inventory_df[['Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

## Section 4: Cybersecurity & Cloud Privacy (PII Masking)

PII means Personally Identifiable Information. Names, contact details, and other identifiers should be protected, minimised, or removed before data is shared or uploaded.

Before patient data is moved to public cloud storage, it should be anonymised or pseudonymised. Hashing can replace names with stable tokens that do not expose the original value.

In [14]:
import hashlib

def anonymize_name(value):
    normalized_value = str(value).strip().lower()
    return hashlib.sha256(normalized_value.encode('utf-8')).hexdigest()

df['Full_Name'] = df['Full_Name'].apply(anonymize_name)
df[['Patient_ID', 'Full_Name']]

,Patient_ID,Full_Name
0,PT-001,94890005f3b2117a353da7260259531878cae4f541bf59...
1,PT-002,a083bb1e6e0e335305d8c73f05df267a9e73d7a0ff6bbe...
2,PT-003,727a69291f7d286125c3fa541dc8f063a0264f9413a9ff...
3,PT-004,02e82576b0769a2397331db8f749d6815cfec92e91b0ae...
4,PT-005,8ec077386d04ed3d0182401b5465002bbcf07379999552...
5,PT-006,fb8bc0d367f5b3bfadcf751a18c5e9aa209cb61bb12f95...
6,PT-007,6c6db149ebbed54df44a59da189a530c396eb78a4e5806...
7,PT-008,e438554de6cf47de7e7bf0c3b0a3c37b33c88d6cb640d0...
8,PT-009,ad617c24c70ec4c78c958dcee192488ed5306a110ace77...
9,PT-010,ad4a5afe7f5090d3e1c1762aaf41c8a5d6aaa83f92d43e...


## Wrap-Up

By the end of the exercises, you should be able to explain how messy inputs are handled, how joins connect business tables, and why privacy controls matter before cloud upload.